In [ ]:
# set up the repo path so imports work from the notebook
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "benchmark").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.chdir(ROOT)
print(f"repo root: {ROOT}")

In [ ]:
# import the benchmark helpers we need
import json

from benchmark.core import (
    load_benchmark_items,
    run_items,
    safe_model_dir_name,
    save_run_outputs,
)
from benchmark.entities import Model
from evaluation.analyze import build_report
from evaluation.evaluate import make_summary
from utils.prompt_builder import build_prompt, load_system_prompt

In [ ]:
# choose your model and run settings here
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DATA_PATH = ROOT / "data" / "dev" / "dev_czech_pub.json"
OUT_DIR = ROOT / "runs"
LIMIT = 10
MAX_TOKENS = 16
TEMPERATURE = 0.0
TOP_P = None
SYSTEM_PROMPT = load_system_prompt()
EXTRA = {}

print("edit model_name if you want to try a different hugging face model")
print(f"model: {MODEL_NAME}")
print(f"dataset: {DATA_PATH}")
print(f"limit: {LIMIT}")
print(f"system prompt loaded: {bool(SYSTEM_PROMPT)}")

In [ ]:
# load the benchmark items and inspect one example prompt
items = load_benchmark_items(DATA_PATH, limit=LIMIT)
sample_item = items[0]
sample_prompt = build_prompt(sample_item)

print(f"loaded {len(items)} items")
print(f"first item id: {sample_item['id']}")
print()
print(sample_prompt)

In [ ]:
# build the hf_local model config
model = Model(
    provider="hf_local",
    model_name=MODEL_NAME,
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    extra=EXTRA,
)

print(model)

In [ ]:
# run the model on the loaded items
results = run_items(
    items=items,
    model=model,
    system_prompt=SYSTEM_PROMPT,
)

print("finished")
print(f"rows: {len(results)}")

In [ ]:
# build the summary from the prediction rows
summary = make_summary(results, model)

print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
# save predictions and summary into runs
run_dir = save_run_outputs(
    results=results,
    summary=summary,
    model=model,
    out_dir=OUT_DIR,
)

print(f"run dir: {run_dir}")
print(f"safe model dir: {safe_model_dir_name(model)}")

In [ ]:
# build a short human-readable analysis from the summary
analysis_text = build_report(summary)

print(analysis_text)

In [ ]:
# save the human-readable analysis next to the run outputs
analysis_path = run_dir / "analysis.txt"
analysis_path.write_text(analysis_text, encoding="utf-8")

print(f"analysis path: {analysis_path}")